In [1]:
# ================================
# INSTALLS
# ================================
!pip install -q \
langchain==1.2.0 \
langchain-core \
langchain-community \
langchain-huggingface \
faiss-cpu \
sentence-transformers \
transformers \
torch

# ================================
# IMPORTS
# ================================
import os
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# ================================
# LOAD LOCAL LLM
# ================================
model_id = "google/flan-t5-base"  # small and works on Colab
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

text_gen_pipeline = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256
)

llm = HuggingFacePipeline(pipeline=text_gen_pipeline)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 103.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 484.9/484.9 kB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Write Query Here: what is pandas


Token indices sequence length is longer than the specified maximum sequence length for this model (1274 > 512). Running this sequence through the model will result in indexing errors



RESULT:
 Python data science ecosystem


In [ ]:

# ================================
# PROMPT: STRICT, FORMAL, 3-7 LINES
# ================================
prompt = PromptTemplate(
    template="""
You are a formal academic assistant.

Answer the question strictly using the provided context only.
Do NOT use prior knowledge or assumptions.
If the context does not fully contain the answer, reply exactly with:

"The answer is not available in the provided documents."

Answering Rules:
- Provide a complete and self-contained answer
- Define each mentioned concept separately (do not merge definitions)
- Answer must be between 3 and 7 complete sentences
- Use clear, formal, and academic language
- Summarize concepts; do NOT copy raw text, metadata, or PDF formatting
- Do NOT give partial or incomplete explanations

Context:
{context}

Question:
{question}

Answer:
""",
    input_variables=["context", "question"]
)

# ================================
# LOAD FAISS VECTORSTORE
# ================================
DB_FAISS_PATH = "db_faiss"

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = FAISS.load_local(
    DB_FAISS_PATH,
    embedding_model,
    allow_dangerous_deserialization=True
)

retriever = db.as_retriever(search_kwargs={"k": 3})

# ================================
# QA PIPELINE
# ================================
qa_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)

# # ================================
# # QUERY
# # ================================
# query = input("Write Query Here: ")
# response = qa_chain.invoke(query)

# # ================================
# # PRINT CLEAN ANSWER ONLY
# # ================================
# print("\nRESULT:\n", response)


Write Query Here: 

RESULT:
 How many pages are in the document?


In [10]:
import gradio as gr

def answer_question(question):
    if not question or question.strip() == "":
        return "Please enter a valid question."

    try:
        response = qa_chain.invoke(question)

        # If response is a dict (some LangChain versions)
        if isinstance(response, dict):
            return response.get("text", str(response))

        return str(response)

    except Exception as e:
        return f"Error: {str(e)}"


with gr.Blocks(title="Medical RAG Chatbot") as demo:
    gr.Markdown(
        """
        ## 🏥 AI Medical Chatbot (RAG)
        **Powered by FLAN-T5 + FAISS + LangChain**

        Ask questions strictly based on uploaded medical documents.
        """
    )

    with gr.Row():
        question_input = gr.Textbox(
            label="Enter your question",
            placeholder="What is pandas?",
            lines=2
        )

    answer_output = gr.Textbox(
        label="Answer",
        lines=6
    )

    submit_btn = gr.Button("Get Answer")

    submit_btn.click(
        fn=answer_question,
        inputs=question_input,
        outputs=answer_output
    )

# IMPORTANT for Hugging Face Spaces
demo.launch(server_name="0.0.0.0", server_port=7860, share=True)


OSError: Cannot find empty port in range: 7860-7860. You can specify a different port by setting the GRADIO_SERVER_PORT environment variable or passing the `server_port` parameter to `launch()`.

/tmp/ipython-input-1425437958.py:2: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


PACKAGE                        STATUS       VERSION
-------------------------------------------------------
gradio                         OK           5.50.0
langchain                      OK           1.2.0
langchain-core                 OK           1.2.5
langchain-community            OK           0.4.1
langchain-huggingface          OK           1.2.0
faiss-cpu                      OK           1.13.2
sentence-transformers          OK           5.2.0
transformers                   OK           4.57.3
torch                          OK           2.9.0+cu126
numpy                          OK           2.0.2
pillow                         OK           11.3.0
